<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation — Theory</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

Segmentation assigns pixels to foreground/background or region labels using image evidence and structural constraints.

### Core Segmentation Model

A segmentation rule maps image features to a mask or label field. Thresholding uses intensity/color decision rules, morphology uses a structuring element to modify binary geometry, and region methods operate on connectivity, distance, or boundary structure.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $I$ | input image |
| $M$ | binary mask |
| $T$ | threshold |
| $B$ | structuring element |
| $G$ | ground-truth mask |
| $P$ | predicted mask |
| IoU | intersection over union |
| Dice | Dice similarity coefficient |

### Analytical Scope

Explain thresholding, morphology, connected components, contours, color/edge segmentation, distance/watershed methods, ground-truth evaluation, failure modes, and strategy selection.


## 1. Segmentation Problem Formulation

Segmentation assigns a label to each pixel.

For binary segmentation:

$$
M(x,y)=
\begin{cases}
1,&\text{foreground}\\
0,&\text{background}
\end{cases}
$$

The result is a **mask**.

A segmentation mask is not merely a visualization. It is a data structure that can be used to:

- measure object area;
- isolate an object;
- compute shape descriptors;
- count instances;
- guide later computer-vision steps.


## 2. Load the Lab Images

This section develops the segmentation theory for load the lab images.


## 3. Histogram-Based Threshold Selection

A grayscale histogram counts how many pixels have each intensity.

If foreground and background occupy different intensity ranges, a threshold can separate them.

Binary thresholding:

$$
M(x,y)=
\begin{cases}
1,&I(x,y)>T\\
0,&I(x,y)\le T
\end{cases}
$$

Whether the foreground is above or below the threshold depends on the image.

### Deeper understanding

For grayscale image $f(x,y)$, threshold segmentation produces

$$
g(x,y)=
\begin{cases}
1,&f(x,y)\in \mathcal{F}\\
0,&\text{otherwise},
\end{cases}
$$

where $\mathcal{F}$ is the intensity set assigned to foreground. A histogram can reveal whether foreground and background form separable intensity populations, but it ignores spatial arrangement. A clear bimodal histogram supports global thresholding; heavy overlap suggests that intensity alone is insufficient.


## 4. Manual Global Thresholding

Let's start with the simplest segmentation.

### Deeper understanding

For scalar threshold $T$,

$$
g(x,y)=
\begin{cases}
1,&f(x,y)\ge T\\
0,&f(x,y)<T.
\end{cases}
$$

The polarity can be reversed when the foreground is darker than the background. The method assumes one decision boundary is valid over the entire image, so it is sensitive to illumination variation and to overlap between class intensity distributions.


## 5. Threshold Sensitivity

A good segmentation method should not be understood from only one threshold.

Let's vary the threshold and observe the effect.

### Deeper understanding

Threshold sensitivity can be viewed through the foreground-area function

$$
A(T)=\sum_{x,y}\mathbf{1}[f(x,y)\ge T].
$$

Rapid changes in $A(T)$ for small changes in $T$ indicate an unstable decision boundary. A robust operating point should preserve the major target regions over a reasonable neighborhood of threshold values rather than depend on one finely tuned scalar.


## 6. Otsu Thresholding

Otsu automatically chooses a global threshold by maximizing separation between two intensity classes.

Equivalent interpretation:

> choose the threshold that minimizes within-class variance.

For candidate threshold $T$:

- pixels below $T$ form one class;
- pixels above $T$ form another class.

Otsu works especially well when the histogram is approximately bimodal.

### Deeper understanding

Otsu's method chooses the threshold that maximizes between-class variance. For class probabilities $\omega_0(T)$ and $\omega_1(T)$ with class means $\mu_0(T)$ and $\mu_1(T)$,

$$
\sigma_B^2(T)=
\omega_0(T)\omega_1(T)
[\mu_0(T)-\mu_1(T)]^2.
$$

The selected threshold is

$$
T^*=\arg\max_T \sigma_B^2(T).
$$

The method works best when the histogram can reasonably be modeled as two separable classes. It does not correct spatially varying illumination.


## 7. Gaussian Smoothing Before Thresholding

Noise can make a binary mask unstable.

A common strategy is:

1. smooth image;
2. threshold smoothed image.

This reduces isolated intensity fluctuations.

### Deeper understanding

Pre-smoothing reduces local noise variance before class assignment. If the Gaussian scale is too small, isolated fluctuations remain; if it is too large, narrow gaps close and true boundaries shift.

The correct smoothing scale should therefore be smaller than the structures that must remain distinct but large enough to suppress nuisance variation.


## 8. Adaptive Thresholding

A single global threshold assumes the same illumination condition everywhere.

Adaptive thresholding computes a local threshold for each neighborhood.

This is useful when illumination changes across the image.

Common forms:

- adaptive mean;
- adaptive Gaussian.

### Deeper understanding

Adaptive thresholding replaces one global threshold with a local decision level,

$$
T(x,y)=m_{\mathcal{N}}(x,y)-C,
$$

where $m_{\mathcal{N}}$ is a local mean or Gaussian-weighted average and $C$ is an offset. This makes the decision relative to local illumination.

Window size sets the illumination scale assumed by the model: too small follows object texture; too large approaches a global threshold and loses local adaptation.


## 9. Morphological Processing

Thresholding frequently creates masks with:

- small false regions;
- small holes;
- rough boundaries;
- disconnected object fragments.

Morphological operations modify binary shapes using a **structuring element**.

The two fundamental operations are:

- erosion;
- dilation.

### Deeper understanding

Mathematical morphology treats a binary image as a set of foreground coordinates and probes it with a structuring element $B$. Operations are therefore geometric and shape-dependent rather than intensity averaging.

The choice of $B$ encodes prior assumptions about relevant scale and orientation. A disk, square, or line structuring element can produce different results even at the same nominal size.


## 10. Structuring Elements

A structuring element defines the neighborhood used by morphology.

Typical shapes:

- rectangle;
- ellipse/disk;
- cross.

Its size determines how aggressively the mask is changed.

### Deeper understanding

A structuring element defines the neighborhood over which morphological inclusion is tested. Its size determines which structures are considered small, while its shape controls directional sensitivity.

For example, a line element can preserve or connect features aligned with that line while suppressing features in other orientations. Structuring-element selection is therefore part of the segmentation model, not a cosmetic parameter.


## 11. Erosion and Dilation

### Erosion

Erosion shrinks foreground regions.

It can:

- remove small bright/foreground noise;
- break thin bridges;
- separate objects.

### Dilation

Dilation expands foreground regions.

It can:

- fill small gaps;
- connect nearby fragments;
- enlarge objects.

### Deeper understanding

For foreground set $A$ and structuring element $B$,

$$
A\ominus B=
\{z:B_z\subseteq A\},
$$

$$
A\oplus B=
\{z:(\hat B)_z\cap A\neq\emptyset\}.
$$

Erosion keeps locations where the translated structuring element fits entirely inside the foreground; dilation keeps locations where it overlaps. These definitions explain why erosion removes thin structures and dilation expands boundaries.


## 12. Opening and Closing

### Opening

Opening = erosion followed by dilation:

$$
A\circ B
=
(A\ominus B)\oplus B
$$

Useful for removing small foreground objects/noise.

### Closing

Closing = dilation followed by erosion:

$$
A\bullet B
=
(A\oplus B)\ominus B
$$

Useful for filling small holes or gaps.


## 13. Morphological Gradient

The morphological gradient approximates object boundaries:

$$
G
=
\text{dilation}
-
\text{erosion}
$$

### Deeper understanding

A morphological gradient is commonly

$$
G_m=(A\oplus B)-(A\ominus B).
$$

It produces a band around object boundaries whose thickness depends on the structuring element. Unlike derivative-based gradients, it operates on set geometry rather than intensity derivatives.


## 14. Hole Filling

A segmented object may contain unwanted internal holes.

SciPy provides a simple binary hole-filling operation.

### Deeper understanding

A hole is a background component completely enclosed by foreground. Hole filling can be formulated through reconstruction from the image border: background connected to the border is retained as true exterior background, while enclosed background components are reclassified as foreground.

This distinction is important because ordinary dilation can also expand the external boundary, whereas true hole filling targets only enclosed cavities.


## 15. Connected Components

Connected-component labeling assigns a unique integer label to each connected foreground region.

For binary image $M$:

- label 0 usually represents background;
- labels 1, 2, 3, ... represent components.

This enables:

- object counting;
- area filtering;
- centroid computation;
- bounding-box extraction.

### Deeper understanding

Connected-component labeling partitions the foreground into maximal connected sets. The result depends on the connectivity definition: in 2-D, 4-connectivity considers horizontal/vertical neighbors, while 8-connectivity also includes diagonals.

Changing connectivity can merge or split diagonally touching structures, so it must be consistent with the intended object topology.


## 16. Remove Small Components

A common cleanup rule is:

> Keep only connected components whose area exceeds a minimum threshold.

This is often safer than blindly applying aggressive morphology.

### Deeper understanding

Area filtering applies a region-level criterion after connected-component labeling:

$$
A_k = |\mathcal{C}_k|.
$$

A component can be retained when $A_k\ge A_{\min}$. This is more interpretable than arbitrary local cleanup because the rejection rule is attached to a measurable object property. The threshold should reflect the smallest valid target size.


## 17. Contours

A contour is an ordered set of boundary points.

Contours are useful for:

- perimeter;
- bounding boxes;
- shape analysis;
- polygon approximation;
- visualization.

### Deeper understanding

A contour is an ordered boundary representation of a connected region. It supports geometric quantities such as perimeter, polygon approximation, convexity, and shape descriptors.

Contours differ from masks: the mask represents region occupancy, while the contour represents only the boundary. Both should be used according to the measurement needed.


## 18. Region Properties

For each contour we can compute useful geometric measurements:

- area;
- perimeter;
- bounding rectangle;
- centroid;
- circularity;
- aspect ratio.

Circularity:

$$
C
=
\frac{4\pi A}{P^2}
$$

A perfect circle has circularity near 1.

### Deeper understanding

For a binary component with area $A$ and pixel coordinates $(x_i,y_i)$, the centroid is

$$
(\bar x,\bar y)=
\left(
\frac{1}{A}\sum_i x_i,
\frac{1}{A}\sum_i y_i
\right).
$$

Bounding boxes, area, centroid, orientation, and compactness convert a pixel mask into object-level measurements. These properties can be used for filtering components or validating that a segmented region has plausible geometry.


## 19. Color Segmentation

Grayscale thresholding ignores color information.

For color images, segmentation can be easier in color spaces such as:

- RGB;
- HSV;
- Lab.

HSV is useful because hue approximately separates color from brightness.

### Deeper understanding

HSV separates hue from saturation and value, which can make chromatic thresholds less sensitive to brightness than raw RGB thresholds. Hue is circular, so colors near the hue origin may require two intervals rather than one contiguous numerical range.

Color segmentation succeeds when chromatic separation is stronger than intensity separation. It remains sensitive to illumination changes that alter saturation/value and to objects whose colors overlap.


## 20. Edge-Based Segmentation

Segmentation can also begin from boundaries.

Typical steps:

1. detect edges;
2. connect broken boundaries;
3. close gaps;
4. fill enclosed regions.

Canny edge detection is a common starting point.

### Deeper understanding

Edges detect transitions, not regions. A boundary detector can therefore provide strong evidence for object borders while still producing an open or fragmented curve.

Turning edges into a region segmentation generally requires linking, contour closure, morphology, or region filling. This explains why edge-based segmentation is rarely complete after the gradient/edge stage alone.


## 21. Distance Transform

For a binary foreground region, the distance transform assigns each foreground pixel its distance to the nearest background pixel.

Inside an object:

- boundary pixels have small distance;
- central pixels have larger distance.

This is extremely useful for separating touching objects.

### Deeper understanding

For foreground set $A$, a distance transform assigns

$$
D(p)=\min_{q\notin A} d(p,q)
$$

to each foreground pixel $p$. Peaks occur near region centers and can serve as markers for separating touching objects.

The chosen metric—Euclidean, city-block, or chessboard—changes the geometry of the distance map and should match the intended spatial interpretation.


## 22. Watershed Segmentation

Watershed treats an image like a topographic surface.

Conceptually:

- low regions behave like basins;
- markers define starting regions;
- the algorithm expands regions until boundaries meet.

Watershed is especially useful for separating touching objects.

A common marker-based workflow is:

1. obtain binary foreground;
2. remove noise;
3. compute sure background;
4. compute distance transform;
5. extract sure foreground;
6. define unknown region;
7. label markers;
8. apply watershed.

### Deeper understanding

Watershed interprets an image or distance surface as a topographic landscape. Flooding from markers grows catchment basins until neighboring basins meet at watershed boundaries.

Marker-controlled watershed is preferred in practice because unconstrained local minima can cause severe over-segmentation. Good markers should lie confidently inside objects and background regions; ambiguous markers directly degrade the final partition.


## 23. Ground Truth and Segmentation Metrics

To evaluate segmentation properly, compare a predicted mask with a ground-truth mask.

Let:

- TP = true positive pixels;
- TN = true negative pixels;
- FP = false positive pixels;
- FN = false negative pixels.

### Deeper understanding

For binary masks, define true positives (TP), false positives (FP), false negatives (FN), and true negatives (TN). Then

$$
\mathrm{Precision}=\frac{TP}{TP+FP},
\qquad
\mathrm{Recall}=\frac{TP}{TP+FN},
$$

$$
\mathrm{IoU}=\frac{TP}{TP+FP+FN},
$$

$$
\mathrm{Dice}=\frac{2TP}{2TP+FP+FN}.
$$

Pixel accuracy can be misleading when background strongly dominates, because a model can classify most background correctly while still missing the object.


## 24. Dice vs IoU Relationship

For the same prediction:

$$
\mathrm{Dice}
=
\frac{2\mathrm{IoU}}
{1+\mathrm{IoU}}
$$

and:

$$
\mathrm{IoU}
=
\frac{\mathrm{Dice}}
{2-\mathrm{Dice}}
$$

Dice is numerically larger than IoU for the same non-perfect segmentation, but both measure overlap.

### Deeper understanding

For non-empty binary masks,

$$
\mathrm{Dice}=
\frac{2\,\mathrm{IoU}}{1+\mathrm{IoU}},
$$

and

$$
\mathrm{IoU}=
\frac{\mathrm{Dice}}{2-\mathrm{Dice}}.
$$

The measures are monotonically related, so they rank segmentations consistently, but their numerical values are not interchangeable. Dice gives relatively more weight to the intersection.


## 25. Under-Segmentation vs Over-Segmentation

### Under-segmentation

Different real objects are merged into one segment.

### Over-segmentation

One real object is split into too many segments.

These concepts are different from false positives and false negatives.

They describe how regions are partitioned.

### Deeper understanding

Under-segmentation merges regions that should be distinct; over-segmentation splits one region into multiple labels. Pixel overlap alone may not fully expose these object-level errors.

The corrective actions differ: under-segmentation may require stronger separation markers or weaker closing, whereas over-segmentation may require smoothing, marker consolidation, or region merging.


## 26. End-to-End Binary Segmentation Pipeline

A practical classical pipeline often looks like:

```text
input image
    ↓
grayscale conversion
    ↓
noise reduction
    ↓
thresholding
    ↓
morphological cleanup
    ↓
hole filling
    ↓
connected components
    ↓
remove small regions
    ↓
contours / measurements
    ↓
final mask + overlay
```

We now package this into a reusable function.

### Deeper understanding

A robust classical pipeline is a sequence of assumptions:

1. preprocessing controls nuisance variation;
2. threshold/color/edge evidence produces an initial mask;
3. morphology enforces local shape regularity;
4. connected components convert pixels to regions;
5. region filtering imposes object-level constraints;
6. metrics compare the result with reference evidence.

Each stage should be justified independently so that a final failure can be traced to the responsible assumption rather than treated as a black box.


## 27. Segmentation Method Selection Criteria

Ask these questions:

### Is intensity separation strong?

Use:

- global threshold;
- Otsu.

### Is illumination uneven?

Try:

- adaptive thresholding;
- illumination correction first.

### Is color the main distinction?

Use:

- HSV;
- Lab;
- color ranges/clustering.

### Are objects touching?

Use:

- distance transform;
- markers;
- watershed.

### Is the mask noisy?

Use:

- opening;
- component-area filtering.

### Are there holes/gaps?

Use:

- closing;
- hole filling.

### Is the task boundary-driven?

Use:

- Canny or other edge detector;
- boundary linking;
- contour processing.


## 28. Integrated Segmentation Workflow

```text
INPUT IMAGE
    │
    ├── intensity based ───────► threshold / Otsu / adaptive
    │
    ├── color based ───────────► HSV / Lab ranges
    │
    └── boundary based ────────► edges / contours
                                │
                                ▼
                         INITIAL MASK
                                │
                                ▼
                       MORPHOLOGICAL CLEANUP
                   erosion / dilation / open / close
                                │
                                ▼
                       CONNECTED COMPONENTS
                                │
                                ▼
                         REGION MEASUREMENTS
                                │
             ┌──────────────────┴──────────────────┐
             │                                     │
      touching objects                        final mask
             │                                     │
       distance transform                           ▼
             │                              Dice / IoU / etc.
         watershed
```

### Deeper understanding

The integrated workflow should preserve **mask semantics** consistently: one value represents foreground and the other background, with no silent polarity changes between operations. Shape, dtype, connectivity, and coordinate conventions should remain explicit.

A trustworthy final result combines visual plausibility, region-level consistency, and quantitative overlap metrics. Passing only one of these checks is insufficient for strong validation.


## Technical Synthesis

The segmentation pipeline converts image evidence into a spatial label field and then validates that label field against structural and reference information:

$$
\boxed{
\text{image}
\rightarrow
\text{preprocessing}
\rightarrow
\text{decision rule}
\rightarrow
\text{morphology/region analysis}
\rightarrow
\text{mask}
\rightarrow
\text{quantitative evaluation}
}
$$

Thresholding, morphology, connected components, color cues, distance transforms, and watershed address distinct failure modes. Selection among them is driven by illumination, class separation, object connectivity, boundary quality, and the availability of ground truth.

## Scope and Limitations

### Included

Thresholding, morphology, connected components, contours/regions, color and edge cues, distance transform, watershed, Dice/IoU, and strategy selection.

### Not included

Learned semantic/instance segmentation networks.


## References

- Gonzalez & Woods, *Digital Image Processing*
- Course slides, Image Processing, Centrale Nantes, 2025–2026
- Official documentation for the libraries used in `main.ipynb`
